# Challenge 1: Building an Agent with Custom Tools

**Goal:** Demonstrate the ability to create and test an agent using the Google Agent Development Kit (ADK).

**Requirements covered in this notebook:**
1. Agent that uses tools to retrieve real-time weather data for user locations.
2. Weather summary based on current conditions and location.
3. Test code demonstrating the agent works for multiple US cities.
4. Custom tool: National Weather Service API (lat/long → weather), with type hints + PEP 8 docstrings.
5. Custom tool: Google Maps Geocoding API (place → lat/long).
6. Tools added to an ADK agent with appropriate instructions.
7. Support for **both** Gemini and a third-party model (via LiteLLM), demonstrated side by side.
8. Uploaded to GitHub for grading.

## Step 0: Setup, Installation, and API Key Management

You'll be prompted for your keys below rather than pasting them into the cell — this keeps real credentials out of the notebook's saved source, so nothing sensitive ends up in git history even if this notebook is re-saved with the prompts already answered. `GOOGLE_MAPS_API_KEY` and `PROJECT_ID` come from the Cloud Skills Boost lab environment. `SAIC_API_KEY` comes from your SAIC-provided LLM gateway credentials.

In [ ]:
!pip install google-adk litellm -q
print("Installation complete.")

In [ ]:
import getpass
import os

# Secrets are prompted for (masked input) rather than hardcoded, so they
# never end up sitting in this cell's saved source.
GOOGLE_MAPS_API_KEY = getpass.getpass("Enter your Google Maps API key: ")
SAIC_API_KEY = getpass.getpass("Enter your SAIC API token: ")

# Not a secret, so a plain prompt is fine.
PROJECT_ID = input("Enter your GCP PROJECT_ID: ")

SAIC_API_BASE = "https://ai-api.apps.factory.saic.com"

os.environ["OPENAI_API_KEY"] = SAIC_API_KEY
os.environ["OPENAI_API_BASE"] = SAIC_API_BASE

print("Environment configured.")

## Step 1: Imports, Settings and Constants

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

# Gemini model (native ADK support, no wrapper needed)
MODEL_GEMINI = "gemini-2.5-flash"

# Third-party model via LiteLLM, routed through the SAIC OpenAI-compatible gateway.
# "bedrock-claude-haiku-4-5" is the cheapest/fastest tier available per SAIC's
# model config — confirm the exact model string matches what the gateway expects.
MODEL_THIRD_PARTY = LiteLlm(model="openai/bedrock-claude-haiku-4-5")

print("Environment configured.")

## Step 2: `get_current_weather(lat, lon)` — National Weather Service Tool

Type-hinted, PEP 8 / PEP 257-style docstring per the assignment requirement.

In [ ]:
import requests


def get_current_weather(lat: float, lon: float) -> str:
    """Retrieve the current weather forecast for a US location.

    Uses the National Weather Service (NWS) API, which requires a two-stage
    lookup: first resolve the (lat, lon) pair to a forecast-office grid
    square via the /points endpoint, then fetch that grid square's
    time-series forecast and return the most immediate period.

    Args:
        lat: Latitude of the target location. Must fall within the United
            States and its territories.
        lon: Longitude of the target location. Must fall within the United
            States and its territories.

    Returns:
        A human-readable summary combining the current forecast period's
        name and detailed forecast text, e.g. "Tonight: Mostly clear, with
        a low around 55." If the NWS API is unavailable or the coordinates
        are out of range, returns a human-readable error message instead
        of raising, so the calling agent can relay it to the user.
    """
    # NWS API requires a descriptive User-Agent header or it returns 403.
    headers = {"User-Agent": "(agent-dev-skills-workshop, jay.watson@saic.com)"}

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]

        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        current_period = forecast_response.json()["properties"]["periods"][0]

        return f"{current_period['name']}: {current_period['detailedForecast']}"
    except requests.RequestException as exc:
        return (
            "The National Weather Service is temporarily unavailable "
            f"(error: {exc}). Please try again in a moment."
        )


# Quick manual check (Washington, DC)
# print(get_current_weather(38.8894, -77.0352))

## Step 3: `get_location_lat_long(city, state)` — Google Maps Geocoding Tool

In [ ]:
def get_location_lat_long(city: str, state: str) -> dict[str, float | None]:
    """Convert a city and state into latitude/longitude coordinates.

    Uses the Google Maps Geocoding API.

    Args:
        city: The city name, e.g. "Knoxville".
        state: The state name or abbreviation, e.g. "TN" or "Tennessee".

    Returns:
        A dict with "Latitude" and "Longitude" keys. Both values are None
        if the location could not be resolved or the API call failed.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": f"{city}, {state}", "key": GOOGLE_MAPS_API_KEY}

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Geocoding API error: {response.status_code}")
        return {"Latitude": None, "Longitude": None}

    data = response.json()
    if data.get("status") != "OK" or not data.get("results"):
        print(f"Geocoding API returned no results: {data.get('status')}")
        return {"Latitude": None, "Longitude": None}

    location = data["results"][0]["geometry"]["location"]
    return {"Latitude": location["lat"], "Longitude": location["lng"]}


# Quick manual check
# print(get_location_lat_long("Knoxville", "TN"))

## Step 4: Build the Weather Agent (model-agnostic)

A factory function so the same agent definition can be instantiated with either Gemini or the third-party model — satisfies requirement #6 without duplicating the instruction/tool wiring.

In [ ]:
AGENT_INSTRUCTION = """
You are a helpful weather assistant, proud through and through to be from
Tennessee.
When the user asks for the weather in a specific city, use the
'get_location_lat_long' function to get the latitude and longitude of the
city, then pass those coordinates to the 'get_current_weather' tool to get
the weather information.
If a tool returns an error, inform the user politely.
If a tool call succeeds, return a clear weather summary.
Any chance you get, work in a mention of how great the Tennessee Volunteers,
Nashville SC, or the Tennessee Titans are - keep it brief and natural, not
forced into every single response, but let your Tennessee pride show.
"""


def build_weather_agent(name: str, model) -> Agent:
    """Build a weather agent instance for the given model.

    Args:
        name: Unique agent name (must differ across instances in the same
            session).
        model: Either a Gemini model string, or a LiteLlm-wrapped model for
            third-party providers.

    Returns:
        A configured ADK Agent with the weather tools attached.
    """
    return Agent(
        name=name,
        model=model,
        description="Provides weather information for specific US cities.",
        instruction=AGENT_INSTRUCTION,
        tools=[get_location_lat_long, get_current_weather],
    )


gemini_agent = build_weather_agent("weather_agent_gemini", MODEL_GEMINI)
third_party_agent = build_weather_agent("weather_agent_third_party", MODEL_THIRD_PARTY)

print(f"Created '{gemini_agent.name}' and '{third_party_agent.name}'.")

## Step 5: Wrap Both Agents, Create Sessions

In [ ]:
from vertexai.preview import reasoning_engines

gemini_app = reasoning_engines.AdkApp(agent=gemini_agent)
third_party_app = reasoning_engines.AdkApp(agent=third_party_agent)

user_id = "test-user-id"
gemini_session = gemini_app.create_session(user_id=user_id)
third_party_session = third_party_app.create_session(user_id=user_id)

print(f"Gemini session: {gemini_session['id']}")
print(f"Third-party session: {third_party_session['id']}")

## Step 6: Query Helper

In [ ]:
def call_weather_agent(app, session_id: str, prompt: str) -> str:
    """Send a prompt to a weather agent app and return its final text reply.

    Args:
        app: An AdkApp instance (either the Gemini or third-party app).
        session_id: The session ID to use for this query.
        prompt: The user's natural-language query.

    Returns:
        The agent's final text response, or a fallback message if none was
        produced.
    """
    response = "sorry, I have no response"
    for event in app.stream_query(user_id=user_id, session_id=session_id, message=prompt):
        content = event.get("content", {})
        parts = content.get("parts", [])
        if parts and "text" in parts[0]:
            response = parts[0]["text"]
    return response

## Step 7: Test — Multiple US Cities, Both Models

Satisfies requirement #3 (multiple US cities) and demonstrates requirement #6 (both models working) in one pass.

In [ ]:
test_cities = [
    "What is the weather in Miami, FL?",
    "What is the weather in Aspen, Colorado?",
    "What is the weather in Knoxville, TN?",
]

print("=== Gemini-backed agent ===\n")
for prompt in test_cities:
    print(f"user: {prompt}")
    print(f"agent: {call_weather_agent(gemini_app, gemini_session['id'], prompt)}\n")

print("=== Third-party-backed agent (via SAIC gateway) ===\n")
for prompt in test_cities:
    print(f"user: {prompt}")
    print(f"agent: {call_weather_agent(third_party_app, third_party_session['id'], prompt)}\n")

## Step 8: Upload to GitHub

Save this notebook and push it to the `agent-dev-skills-workshop-jay-watson` repository for grading.